# HeatMap of the S&P 400 - Mid Capitalization Stocks
**By Paranj Patel & Ved Patel**

**The Problem:**
An investor looking at the mid-capitalization segment of the US Stock Market has no quick way to see where today's movement was concentrated. A list of 400 tickers with their daily return would be unreadable. Sector-level averages hide the outliers. Our heatmap will be a solution to both.

**The Vision:** Each rectangle is one company, the rectangles will be grouped together by their GICS sector classification.


Setup

In [1]:
%pip install -q yfinance plotly

In [3]:
import time
import io
from getpass import getpass

import requests
import pandas as pd
import yfinance as yf
import plotly.express as px

print("yfinance", yf.__version__)

yfinance 0.2.66


Can we get a current, machine-readable list of all 400 S&P MidCap 400 constituents with their sector?

In [4]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies"
html = requests.get(url, headers={"User-Agent": "student-project"}).text

companies = pd.read_html(io.StringIO(html))[0]
companies = companies[["Symbol", "GICS Sector"]]

print(companies.shape)
companies.head()

(400, 2)


,Symbol,GICS Sector
0,AA,Materials
1,AAL,Industrials
2,AAON,Industrials
3,ACI,Consumer Staples
4,ACM,Industrials


In [6]:
#Wikipedia and Yahoo Finance write the tickers for share classes differently, as of now, only 1 stock is affected (MOG.A).
#Index membership changes periodically so we used the below code instead

odd = [s for s in companies["Symbol"] if not str(s).isalpha()]
print("symbols needing the fix:", odd)

companies["Symbol"] = companies["Symbol"].str.replace(".", "-", regex=False)
tickers = companies["Symbol"].tolist()

print(f"\n{len(tickers)} tickers, {companies['GICS Sector'].nunique()} sectors")
print(companies["GICS Sector"].value_counts())

symbols needing the fix: ['MOG.A']

400 tickers, 11 sectors
GICS Sector
Industrials               87
Financials                65
Consumer Discretionary    57
Information Technology    49
Health Care               35
Real Estate               28
Materials                 26
Energy                    17
Consumer Staples          15
Utilities                 15
Communication Services     6
Name: count, dtype: int64


Can we get daily price data for all ~400 tickers?

In [8]:
from google.colab import userdata
AV_KEY = userdata.get('ALPHAVANTAGE_API_KEY')
AV_URL = "https://www.alphavantage.co/query"
r = requests.get(AV_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": "AAON", "apikey": AV_KEY
    })
print("HTTP status:", r.status_code)
r.json()

HTTP status: 200


{'Global Quote': {'01. symbol': 'AAON',
  '02. open': '91.7500',
  '03. high': '95.8700',
  '04. low': '90.2401',
  '05. price': '94.8300',
  '06. volume': '1297924',
  '07. latest trading day': '2026-08-07',
  '08. previous close': '90.2800',
  '09. change': '4.5500',
  '10. change percent': '5.0399%'}}

In [9]:
r = requests.get(AV_URL, params={
    "function": "REALTIME_BULK_QUOTES",
    "symbol": "AAON,AA,AAL",
    "apikey": AV_KEY,
})
print("HTTP status:", r.status_code)
r.json()

HTTP status: 200


{'endpoint': 'Realtime Bulk Quotes',
 'message': "This is a premium endpoint. ***THE SAMPLE DATA SCHEMA BELOW IS ARTIFICIAL AND FOR ILLUSTRATION PURPOSES ONLY***. To access the actual data, please subscribe to any premium plan that mentions 'Realtime US Market Data' in its description at https://www.alphavantage.co/premium/ for your personal non-professional use. For professional/commercial use, please contact support at support@alphavantage.co.",
 'data': [{'symbol': 'MSFT',
   'timestamp': '2024-10-18 19:59:55.291',
   'open': '417.61',
   'high': '419.649',
   'low': '416.2601',
   'close': '418.16',
   'volume': '17145307',
   'previous_close': '416.72',
   'change': '1.44',
   'change_percent': '0.3456',
   'extended_hours_quote': '418.1',
   'extended_hours_change': '-0.06',
   'extended_hours_change_percent': '-0.01435'},
  {'symbol': 'AAPL',
   'timestamp': '2024-10-18 19:59:50.451',
   'open': '236.0',
   'high': '236.05',
   'low': '234.02',
   'close': '235.0',
   'volume': 

Unfortunately, we cannot use AlphaVantage as it requires a premium key, further research shows that even if we had a premium key, they only accept upto 100 ticker requests at a time. Thus, we pivoted to using Yahoo Finance.

In [10]:
start = time.time()
data = yf.download(tickers, period="10d", auto_adjust=True, progress=False)
yf_elapsed = time.time() - start

close = data["Close"]
print(f"runtime: {yf_elapsed:.0f}s")
print("shape:", close.shape)
print("dates:", list(close.index.date))

runtime: 40s
shape: (10, 400)
dates: [datetime.date(2026, 7, 27), datetime.date(2026, 7, 28), datetime.date(2026, 7, 29), datetime.date(2026, 7, 30), datetime.date(2026, 7, 31), datetime.date(2026, 8, 3), datetime.date(2026, 8, 4), datetime.date(2026, 8, 5), datetime.date(2026, 8, 6), datetime.date(2026, 8, 7)]


In [11]:
# Coverage audit - this number decides whether the idea works.
all_nan = close.columns[close.isna().all()].tolist()
usable = close.shape[1] - len(all_nan)

print(f"requested: {len(tickers)}")
print(f"usable:    {usable}  ({usable/len(tickers):.1%} coverage)")
print(f"empty:     {len(all_nan)} -> {all_nan}")

requested: 400
usable:    400  (100.0% coverage)
empty:     0 -> []


Can the two datasets be joined?

In [12]:
returns = (close.iloc[-1] / close.iloc[-2] - 1) * 100
returns = returns.dropna()

df = pd.DataFrame({"Symbol": returns.index, "Return": returns.values})
df = df.merge(companies, on="Symbol", how="left")

print(f"{len(df)} companies, {df['GICS Sector'].isna().sum()} missing a sector")
df.head()

400 companies, 0 missing a sector


,Symbol,Return,GICS Sector
0,AA,5.665541,Materials
1,AAL,-0.561454,Industrials
2,AAON,5.039879,Industrials
3,ACI,-0.579468,Consumer Staples
4,ACM,1.214462,Industrials


In [13]:
# Sanity check before trusting any picture drawn from this.
print(df["Return"].describe().round(2))
print("\nbiggest movers:")
print(df.nlargest(3, "Return")[["Symbol", "GICS Sector", "Return"]])
print(df.nsmallest(3, "Return")[["Symbol", "GICS Sector", "Return"]])
print("\nmean by sector:")
print(df.groupby("GICS Sector")["Return"].mean().round(2).sort_values())

count    400.00
mean       1.20
std        3.56
min      -12.79
25%       -0.40
50%        0.66
75%        2.14
max       32.62
Name: Return, dtype: float64

biggest movers:
    Symbol             GICS Sector     Return
107   DOCS             Health Care  32.623426
356   TWLO  Information Technology  24.886130
166   HALO             Health Care  20.242538
    Symbol             GICS Sector     Return
281   POST        Consumer Staples -12.789538
116   ELAN             Health Care  -8.595041
26     ARW  Information Technology  -8.448423

mean by sector:
GICS Sector
Energy                   -0.47
Financials               -0.29
Real Estate               0.63
Communication Services    0.77
Utilities                 0.92
Consumer Discretionary    1.06
Industrials               1.28
Consumer Staples          1.86
Materials                 2.19
Health Care               2.41
Information Technology    2.61
Name: Return, dtype: float64


In [16]:
# values= sets each box's area. Everything is 1 for now, so all boxes are equal.
# Sizing by market cap would be better but costs one extra request per company.
df["Size"] = 1

fig = px.treemap(
    df,
    path=["GICS Sector", "Symbol"],
    values="Size",
    color="Return",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    range_color=[-3, 3],   # clamp so one outlier doesn't flatten the whole scale
    title=f"S&P MidCap 400 Daily Return by Sector — {close.index[-1].date()}",
)
fig.update_layout(height=1000)
fig.show()